<a href="https://colab.research.google.com/github/ohki-yu0225/toukei_I/blob/main/261113_5_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Data

The data used in this notebook are J1 League match records obtained from the [match records and data search](https://data.j-league.or.jp/SFMS01/) page of the J.LEAGUE Data Site, the official J.League website.

- Source: https://data.j-league.or.jp/SFMS01/
- `2017-2025年J1試合記録_9チーム.csv` contains the home matches, from 2017 to 2025 (excluding 2020 and 2021), of the nine teams that stayed in J1 throughout this period.

In [ ]:
import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (4, 3)
%config InlineBackend.figure_format = "retina"

file_id = "1MSihI6mLVVGaq-Xk8LwU-9Kkzyqf9Aoz"

gdown.download(
    id=file_id,
    output="2017-2025年J1試合記録_9チーム.csv",
    quiet=False
)

df = pd.read_csv("2017-2025年J1試合記録_9チーム.csv", encoding="utf-8-sig")

# Keep the columns used in this notebook and rename them in English
df = df[["シーズン", "試合日", "曜日", "ホーム", "ホーム得点", "アウェイ得点", "入場者数"]]
df.columns = ["Season", "Date", "Weekday", "Home", "HomeGoals", "AwayGoals", "Attendance"]

# Translate team names
df["Home"] = df["Home"].replace({
    "横浜FM": "Yokohama F. Marinos", "鹿島": "Kashima Antlers", "広島": "Sanfrecce Hiroshima",
    "Ｃ大阪": "Cerezo Osaka", "Ｇ大阪": "Gamba Osaka", "浦和": "Urawa Reds",
    "FC東京": "FC Tokyo", "神戸": "Vissel Kobe", "川崎Ｆ": "Kawasaki Frontale"
})

# Translate weekdays (e.g. "土・祝" = Saturday and national holiday, "月・休" = Monday and substitute holiday)
df["Holiday"] = df["Weekday"].str.contains("祝|休")
df["Weekday"] = df["Weekday"].str[0].replace({
    "月": "Mon", "火": "Tue", "水": "Wed", "木": "Thu", "金": "Fri", "土": "Sat", "日": "Sun"
})
df

In [ ]:
# Compare home goals and away goals to determine whether the home team won, drew, or lost, and add it to df
df["HomeResult"] = np.where(df["HomeGoals"] > df["AwayGoals"], "Win",
                   np.where(df["HomeGoals"] == df["AwayGoals"], "Draw", "Loss"))

# Add a column grouping home goals into "0", "1", "2", and "3 or more"
df["HomeGoalGroup"] = np.where(df["HomeGoals"] >= 3, "3 or more", df["HomeGoals"].astype(str))

# Group matches on national holidays and substitute holidays as "Holiday"
df["DayType"] = np.where(df["Holiday"], "Holiday", df["Weekday"])
df

### Frequency Distribution and Histogram

In [ ]:
# Create a frequency distribution of attendance
class_width = 10000
bins = np.arange(0, df["Attendance"].max() + class_width, class_width)
freq = pd.cut(df["Attendance"], bins=bins, right=False).value_counts().sort_index()
freq_table = pd.DataFrame({
    "Frequency": freq,
    "RelativeFrequency": (freq / freq.sum()).round(3),
    "CumulativeFrequency": freq.cumsum(),
})
freq_table

In [ ]:
# Draw a histogram of attendance
class_width = 10000
bins = np.arange(0, df["Attendance"].max() + class_width, class_width)
plt.hist(df["Attendance"], bins=bins, edgecolor="black")
plt.xlabel("Attendance (people)")
plt.ylabel("Frequency (matches)")
plt.title("Histogram of Attendance")
plt.show()

In [ ]:
# Exercise: Change the class width and create a frequency distribution
class_width = 10000
bins = np.arange(0, df["Attendance"].max() + class_width, class_width)
freq = pd.cut(df["Attendance"], bins=bins, right=False).value_counts().sort_index()
freq_table = pd.DataFrame({
    "Frequency": freq,
    "RelativeFrequency": (freq / freq.sum()).round(3),
    "CumulativeFrequency": freq.cumsum(),
})
freq_table

In [ ]:
# Exercise: Change the class width and draw a histogram
class_width = 10000
bins = np.arange(0, df["Attendance"].max() + class_width, class_width)
plt.hist(df["Attendance"], bins=bins, edgecolor="black")
plt.xlabel("Attendance (people)")
plt.ylabel("Frequency (matches)")
plt.title("Histogram of Attendance")
plt.show()

### Measures of Central Tendency and Spread

In [ ]:
# Calculate the mean and median of attendance
mean = df["Attendance"].mean()
median = df["Attendance"].median()
print(f"Mean: {mean:.1f} people")
print(f"Median: {median:.1f} people")

In [ ]:
# Calculate the standard deviation and interquartile range of attendance
std = df["Attendance"].std()
q1 = df["Attendance"].quantile(0.25)
q3 = df["Attendance"].quantile(0.75)
iqr = q3 - q1
print(f"Standard deviation: {std:.1f} people")
print(f"First quartile: {q1:.1f} people, Third quartile: {q3:.1f} people")
print(f"Interquartile range: {iqr:.1f} people")

### Box Plot

In [ ]:
# Draw a box plot of attendance
sns.boxplot(y=df["Attendance"])
plt.ylabel("Attendance (people)")
plt.ylim(0, None)
plt.title("Box Plot of Attendance")
plt.show()

In [ ]:
# Draw box plots of attendance by team
sns.boxplot(data=df, x="Home", y="Attendance", order=None)
plt.xticks(rotation=90)
plt.xlabel("Home team")
plt.ylabel("Attendance (people)")
plt.ylim(0, None)
plt.title("Attendance by Team (Home Matches)")
plt.show()

In [ ]:
# Exercise: Draw box plots of attendance by day of the week
order = [d for d in ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun", "Holiday"] if d in df["DayType"].unique()]

### Cross Tabulation

In [ ]:
# Create a cross tabulation of home wins, draws, and losses by team
cross = pd.crosstab(df["Home"], df["HomeResult"])
cross

In [ ]:
# Normalize so that each row sums to 100% (win, draw, and loss rates by team)
cross_rate = pd.crosstab(df["Home"], df["HomeResult"], normalize="index")
cross_rate

In [ ]:
# Visualize the cross tabulation as a heatmap
sns.heatmap(cross_rate, annot=True, fmt=".3f", cmap="Blues")
plt.xlabel("Home result")
plt.ylabel("Home team")
plt.title("Home Results by Team")
plt.show()

In [ ]:
# Exercise: Create a cross tabulation of home goals by team

In [ ]:
# Exercise: Normalize so that each row sums to 100%

In [ ]:
# Exercise: Visualize the cross tabulation as a heatmap

### Scatter Plot and Correlation Coefficient

In [ ]:
# Calculate the home win rate and attendance by team and season
team_stats = df.groupby(["Home", "Season"]).agg(
    Matches=("HomeResult", "size"),
    HomeWinRate=("HomeResult", lambda x: (x == "Win").mean()),
    MeanAttendance=("Attendance", "mean"),
)

# Add each team's mean and the deviation of each season's value from the team mean
team_stats["TeamMeanHomeWinRate"] = team_stats.groupby(level="Home")["HomeWinRate"].transform("mean")
team_stats["TeamMeanAttendance"] = team_stats.groupby(level="Home")["MeanAttendance"].transform("mean")
team_stats["HomeWinRateDev"] = team_stats["HomeWinRate"] - team_stats["TeamMeanHomeWinRate"]
team_stats["MeanAttendanceDev"] = team_stats["MeanAttendance"] - team_stats["TeamMeanAttendance"]
team_stats

In [ ]:
# Visualize the relationship between home win rate and mean attendance with a scatter plot
plt.scatter(team_stats["HomeWinRate"], team_stats["MeanAttendance"])
plt.xlabel("Home win rate")
plt.ylabel("Mean attendance (people)")
plt.title("Home Win Rate and Mean Attendance")
plt.show()

In [ ]:
# Calculate the correlation coefficient between home win rate and mean attendance
r = team_stats["HomeWinRate"].corr(team_stats["MeanAttendance"])
print(f"Correlation coefficient: {r:.3f}")

In [ ]:
# Exercise: Visualize the deviations from each team's mean with a scatter plot

In [ ]:
# Exercise: Calculate the correlation coefficient of the deviations from each team's mean